In [ ]:
#   5.1 制作数据集
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
%matplotlib inline

# 展示高清图
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats('svg')

#   在封装我们的数据集时，必须继承实用工具(utils)中的DataSet的类，这
# 个过程需要重写__init__、__getitem__、__len__三个方法，分别是为了加载数据
# 集、获取数据索引、获取数据总量。

class MyData(Dataset):      # 继承Dataset类
    def __init__(self, filepath):
        df = pd.read_csv(filepath, index_col=0)
        arr = df.values                 # 对象退化为数组
        arr = arr.astype(np.float32)    # 转为float32类型数组
        ts = torch.tensor(arr)          # 数组转为张量
        ts = ts.to('cuda')              # 把训练集搬到cuda上
        self.X = ts[:, :-1]                 # 前8列为输入特征
        self.Y = ts[:, -1].reshape((-1,1))  # 后1列为输出特征
        self.len = ts.shape[0]              # 样本的总数

    def __getitem__(self, index):
        return self.X[index], self.Y[index]

    def __len__(self):
        return self.len

# 划分训练集与测试集
Data = MyData('Data.csv')
train_size = int(len(Data) * 0.8)
test_size = len(Data) - train_size
train_Data, test_Data = random_split(Data, [train_size, test_size])

# 批次加载器
train_loader = DataLoader(dataset=train_Data, batch_size=128, shuffle=True)     # shuffle用于在每个epoch内先洗牌再分批
test_loader = DataLoader(dataset=test_Data, batch_size=64, shuffle=False)

In [ ]:
#   5.2 搭建神经网络
class DNN(nn.Module):

    def __init__(self):
        '''搭建神经网络各层'''
        super(DNN, self).__init__()
        self.net = nn.Sequential(           # 按顺序搭建各层
            nn.Linear(8, 32), nn.Sigmoid(),
            nn.Linear(32, 8), nn.Sigmoid(),
            nn.Linear(8, 4), nn.Sigmoid(),
            nn.Linear(4, 1), nn.Sigmoid()
        )

    def forward(self, x):
        '''前向传播'''
        y = self.net(x)     # x即输入数据
        return y            # y即输出数据

model = DNN()       # 创建子类的实例
# model = DNN().to('cuda:0')    # 搬到GPU上
print(model)

In [ ]:
#   5.3 训练网络

# 损失函数的选择
loss_fn = nn.BCELoss(reduction='mean')

# 优化算法的选择
learning_rate = 0.005   # 设置学习率
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

epochs = 500
losses = []     # 记录损失函数变化的列表

for epoch in range(epochs):
    for (x, y) in train_loader:     # 获取小批次的x与y
        Pred  =model(x)             # 一次前向传播（MBGD）
        loss = loss_fn(Pred, y)     # 计算损失函数
        losses.append(loss.item())  # 记录损失函数的变化
        optimizer.zero_grad()       # 清理上一轮滞留的梯度
        loss.backward()             # 一次反向传播
        optimizer.step()            # 优化内部参数

Fig = plt.figure()
plt.plot(range(len(losses)), losses)
plt.show()

In [ ]:
#   5.4 测试网络

correct = 0
total = 0

with torch.no_grad():       # 关闭梯度计算功能
    for x, y in test_loader:    # 获取小批次的x与y
        Pred =model(x)          # 一次前向传播(小批量)
        Pred[Pred >= 0.5] = 1
        Pred[Pred < 0.5] = 0
        correct = torch.sum((Pred==y).all(1))   # 预测正确的样本
        total = y.size(0)                       # 全部的样本数量

print(f'测试集精准度:{100*correct/total} % ')